[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch14-ml-ops/02-khoang-huan-luyen-lai.ipynb)

# Bao lâu thì huấn luyện lại: một bất đẳng thức, một đường chữ U và hai chính sách

Pipeline huấn luyện tự động đặt ra một quyết định then chốt: chạy nó bao lâu một lần. Chương 14 trả
lời theo ba tầng. Phương trình 3 là một điều kiện để cân nhắc: chỉ huấn luyện lại khi lợi ích kỳ vọng
vượt chi phí lần chạy cộng rủi ro triển khai. Phương trình 5 tới 8 dựng một mô hình kinh tế, và
napkin math 1.3 rút ra từ đó khoảng huấn luyện lại tối ưu $T^*$. Hình 8 đặt hai chính sách cạnh
nhau: huấn luyện lại theo lịch, và huấn luyện lại khi chất lượng chạm ngưỡng.

Sổ tay này tính lại cả ba, rồi kiểm những chỗ sách tự nói là đơn giản hoá: xấp xỉ căn bậc hai sai
bao nhiêu so với cực tiểu chính xác, chính sách kích hoạt còn giữ được mức sàn khi nhãn về trễ hay
không, và một pipeline tự động không người duyệt có thể đưa một mô hình tồi vào sản xuất thế nào.

**Bạn sẽ làm:**
1. đổi phương trình 3 thành một phép thử bằng tiền, với các tham số của bảng 9;
2. vẽ độ chính xác suy giảm theo hàm mũ, tính chi phí lỗi thời bằng dạng đóng và bằng tích phân số;
3. dựng đường chữ U của chi phí mỗi ngày, tìm $T^*$ bằng xấp xỉ của sách và bằng cực tiểu chính xác;
4. kiểm bảng 10, rồi tìm ra một đường cong duy nhất quyết định khi nào xấp xỉ còn đúng;
5. mô phỏng hình 8 với nhãn về trễ và nhãn nhiễu;
6. chạy một pipeline huấn luyện lại tự động trên dữ liệu "cuối tuần" và xem cổng kiểm định nào chặn
   được nó.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
from scipy import integrate, optimize
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định
XAM = "0.45"

# Bảng 9: bài toán gian lận của napkin math 1.3.
Q = sach(1_000_000, nguon="Q = 1,000,000")        # giao dịch mỗi ngày
V = sach(0.50, nguon="V = 0.50 đô la")            # đô la mỗi truy vấn cho một đơn vị độ chính xác
ACC0 = sach(0.95, nguon="Acc₀ = 0.95")            # độ chính xác ban đầu
GAMMA = sach(0.02, nguon="γ = 0.02")              # tốc độ suy giảm mỗi ngày
C = sach(5_000, nguon="C = 5,000 đô la")          # chi phí một lần huấn luyện lại
print(f"Q = {Q:,.0f}/ngày · V = {V} đô la · Acc0 = {ACC0} · γ = {GAMMA}/ngày · C = {C:,} đô la")

## 1. Phương trình 3: một điều kiện, không phải một công tắc

$$\text{huấn luyện lại nếu: } \Delta\text{độ chính xác} \times \text{giá trị một điểm} > \text{chi phí huấn luyện} + \text{rủi ro triển khai}$$

Vế trái là mức tăng độ chính xác kỳ vọng, tính bằng điểm phần trăm, nhân giá trị tiền của một điểm
trong khung thời gian ra quyết định. Với quy ước của napkin math 1.3, $V = 0.50$ đô la nghĩa là một
điểm phần trăm đáng 0.005 đô la mỗi truy vấn. Ô dưới đổi điều đó ra tiền mỗi ngày, rồi thử vài mức
tăng và vài khung thời gian. Rủi ro triển khai 2,000 đô la là giả định của sổ tay.

In [ ]:
MOT_DIEM = V / 100
theo_sach("một điểm phần trăm, mỗi truy vấn (đô la)", MOT_DIEM, sach=0.005, nguon="0.005 đô la")
mot_diem_ngay = MOT_DIEM * Q
print(f"một điểm phần trăm, cả ngày: {mot_diem_ngay:,.0f} đô la")

RUI_RO = 2_000  # đô la, giả định của sổ tay
print(f"\nvế phải: {C:,} + {RUI_RO:,} = {C + RUI_RO:,} đô la")
print("tăng (điểm)  khung 1 ngày   khung 7 ngày   khung 30 ngày")
for tang in (0.1, 0.5, 1.0, 2.0):
    o = []
    for khung in (1, 7, 30):
        loi = tang * mot_diem_ngay * khung
        o.append(f"{loi:>9,.0f} {'✓' if loi > C + RUI_RO else '·'}")
    print(f"{tang:>8.1f}     " + "    ".join(o))

Ở lưu lượng 1 triệu giao dịch mỗi ngày, một điểm phần trăm đáng 5,000 đô la mỗi ngày, đúng bằng chi
phí một lần huấn luyện lại. Vì thế ngay cả một mức tăng nhỏ cũng vượt điều kiện nếu nó giữ được vài
ngày. Sách vẫn nhấn mạnh đây là điều kiện để cân nhắc: chỉ huấn luyện lại khi **bằng chứng về kết
quả** cho thấy lợi ích kỳ vọng ấy là thật, không phải chỉ vì một tín hiệu trôi vừa bật.

## 2. Độ chính xác suy giảm, và chi phí lỗi thời

Phương trình 5 là một đường khớp hàm mũ cục bộ, mà sách gọi là một cách đơn giản hoá để có lời giải
dạng đóng:

$$\text{Acc}(t) = \text{Acc}_0 \cdot e^{-\gamma t}$$

Phương trình 6 lấy tích phân phần độ chính xác bị mất trên toàn khoảng huấn luyện lại $T$:

$$\text{chi phí lỗi thời}(T) = \int_0^T Q V \big(\text{Acc}_0 - \text{Acc}(t)\big)\,dt = Q V \text{Acc}_0 \left(T - \frac{1 - e^{-\gamma T}}{\gamma}\right)$$

**Dự đoán:** với $\gamma = 0.02$ mỗi ngày, độ chính xác khớp được còn một nửa sau bao lâu? Khoảng
huấn luyện lại tối ưu dài bằng chừng ấy, ngắn hơn nhiều, hay dài hơn?

In [ ]:
def do_chinh_xac(t, acc0=ACC0, gamma=GAMMA):
    """Phương trình 5."""
    return acc0 * np.exp(-gamma * np.asarray(t, dtype=float))


def chi_phi_loi_thoi(T, q=Q, v=V, acc0=ACC0, gamma=GAMMA):
    """Phương trình 6, dạng đóng. expm1 giữ độ chính xác khi γT nhỏ."""
    T = np.asarray(T, dtype=float)
    return q * v * acc0 * (T + np.expm1(-gamma * T) / gamma)


ban_ra = np.log(2) / GAMMA
print(f"chu kỳ bán rã của đường khớp: ln 2 / γ = {ban_ra:.1f} ngày")

# Dạng đóng phải khớp tích phân số của chính định nghĩa.
for T in (0.5, 1.0, 7.0, 30.0):
    so, _ = integrate.quad(lambda t: Q * V * (ACC0 - do_chinh_xac(t)), 0, T)
    dong = float(chi_phi_loi_thoi(T))
    assert abs(so - dong) <= 1e-9 * dong
    print(f"T = {T:>4} ngày: chi phí lỗi thời {dong:>12,.1f} đô la (tích phân số khớp tới 1e-9)")

In [ ]:
t = np.linspace(0, 120, 400)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(t, do_chinh_xac(t), color=MAU[0], lw=2.2)
ax1.axhline(ACC0 / 2, color=XAM, lw=0.9, ls=":")
ax1.axvline(ban_ra, color=XAM, lw=0.9, ls=":")
ax1.annotate(f"còn một nửa sau {ban_ra:.1f} ngày", (ban_ra, ACC0 / 2), xytext=(12, 22), textcoords="offset points",
             fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax1.set_xlabel("ngày kể từ lần huấn luyện gần nhất")
ax1.set_ylabel("độ chính xác khớp được")
ax1.set_title("Phương trình 5: Acc0 = 0.95, γ = 0.02 mỗi ngày")
ax1.set_ylim(0, 1)
T_ve = np.linspace(0, 10, 200)
ax2.plot(T_ve, chi_phi_loi_thoi(T_ve) / 1e3, color=MAU[1], lw=2.2)
ax2.plot(T_ve, Q * V * ACC0 * GAMMA * T_ve ** 2 / 2 / 1e3, color=MAU[1], lw=1.2, ls="--")
ax2.annotate("dạng đóng", (8.5, float(chi_phi_loi_thoi(8.5)) / 1e3), xytext=(8.0, 120), fontsize=8, color="0.25",
             ha="center", arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax2.annotate("xấp xỉ bậc hai: QVAcc0·γT²/2", (7.5, Q * V * ACC0 * GAMMA * 7.5 ** 2 / 2 / 1e3), xytext=(1.0, 330),
             fontsize=8, color="0.25", arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax2.set_xlabel("khoảng huấn luyện lại T (ngày)")
ax2.set_ylabel("chi phí lỗi thời trong khoảng (nghìn đô la)")
ax2.set_title("Phương trình 6: gần như bậc hai khi γT nhỏ")
plt.tight_layout()
plt.show()

Chu kỳ bán rã hơn một tháng, nhưng đó không phải câu trả lời. Như sách nói, chu kỳ bán rã cho biết
khi nào độ chính xác khớp được còn một nửa, chứ không quyết định khoảng huấn luyện lại tối ưu về
kinh tế. Hình bên phải cho thấy vì sao có một xấp xỉ: khi $\gamma T$ nhỏ, chi phí lỗi thời tăng gần
như theo $T^2$.

## 3. Đường chữ U, và khoảng huấn luyện lại tối ưu

Phương trình 8 định nghĩa $T^*$ là khoảng làm cực tiểu tổng chi phí trên mỗi đơn vị thời gian:

$$T^* = \arg\min_T \frac{\text{chi phí lỗi thời}(T) + \text{chi phí huấn luyện lại}}{T}$$

Huấn luyện lại càng dày thì phần chi phí huấn luyện chia cho mỗi ngày càng lớn; càng thưa thì phần
lỗi thời mỗi ngày càng lớn. Khi $\gamma T \ll 1$, khai triển Taylor cho phương trình 4, xấp xỉ căn
bậc hai mà napkin math 1.3 dùng:

$$T^* \approx \sqrt{\frac{2C}{Q \cdot V \cdot \text{Acc}_0 \cdot \gamma}}$$

Sách nói muốn có cực tiểu chính xác thì phải cực tiểu hoá phương trình 8 mà không dùng khai triển.
Ô dưới làm cả hai.

In [ ]:
def chi_phi_moi_ngay(T, c=C, q=Q, v=V, acc0=ACC0, gamma=GAMMA):
    """Phương trình 8, phần trong arg min."""
    return (chi_phi_loi_thoi(T, q, v, acc0, gamma) + c) / np.asarray(T, dtype=float)


def T_xap_xi(c=C, q=Q, v=V, acc0=ACC0, gamma=GAMMA):
    """Phương trình 4."""
    return np.sqrt(2 * c / (q * v * acc0 * gamma))


def T_chinh_xac(c=C, q=Q, v=V, acc0=ACC0, gamma=GAMMA):
    """Cực tiểu số của phương trình 8, tìm theo log T."""
    kq = optimize.minimize_scalar(lambda u: chi_phi_moi_ngay(np.exp(u), c, q, v, acc0, gamma),
                                  bounds=(np.log(1e-4), np.log(1e4)), method="bounded", options={"xatol": 1e-12})
    return float(np.exp(kq.x))


T_a, T_e = T_xap_xi(), T_chinh_xac()
theo_sach("T* theo xấp xỉ căn bậc hai (ngày)", T_a, sach=1, nguon="T* ≈ 1 ngày", tol=0.05)
print(f"T* chính xác, cực tiểu phương trình 8: {T_e:.4f} ngày; xấp xỉ lệch {T_a / T_e - 1:+.2%}")
print(f"chi phí mỗi ngày ở T* chính xác: {float(chi_phi_moi_ngay(T_e)):,.0f} đô la;"
      f" ở T* xấp xỉ: {float(chi_phi_moi_ngay(T_a)):,.0f} đô la")
for ten, T in (("hằng ngày", 1), ("hằng tuần", 7), ("hằng tháng", 30)):
    print(f"  {ten:<10} T = {T:>2}: {float(chi_phi_moi_ngay(T)):>9,.0f} đô la mỗi ngày,"
          f" gấp {float(chi_phi_moi_ngay(T) / chi_phi_moi_ngay(T_e)):.2f} lần mức tối ưu")

In [ ]:
T_ve = np.logspace(np.log10(0.05), np.log10(60), 400)
phan_hl = C / T_ve
phan_lt = chi_phi_loi_thoi(T_ve) / T_ve
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(T_ve, (phan_hl + phan_lt) / 1e3, color="0.15", lw=2.4)
ax.plot(T_ve, phan_hl / 1e3, color=MAU[0], lw=1.6, ls="--")
ax.plot(T_ve, phan_lt / 1e3, color=MAU[1], lw=1.6, ls="--")
ax.text(0.07, C / 0.07 / 1e3 * 0.62, "chi phí huấn luyện lại / T", color=MAU[0], fontsize=8)
ax.text(12, float(chi_phi_loi_thoi(12)) / 12 / 1e3 * 0.55, "chi phí lỗi thời / T", color=MAU[1], fontsize=8)
ax.plot([T_e], [float(chi_phi_moi_ngay(T_e)) / 1e3], "o", color="0.15", ms=8, zorder=5)
ax.annotate(f"T* = {T_e:.2f} ngày", (T_e, float(chi_phi_moi_ngay(T_e)) / 1e3), xytext=(1.0, 2.2),
            fontsize=8, ha="center", arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
for T, ten in ((7, "hằng tuần"), (30, "hằng tháng")):
    ax.plot([T], [float(chi_phi_moi_ngay(T)) / 1e3], "o", color="0.15", ms=5)
    ax.annotate(f"{ten}: {float(chi_phi_moi_ngay(T) / chi_phi_moi_ngay(T_e)):.1f} lần", (T, float(chi_phi_moi_ngay(T)) / 1e3),
                xytext=(-8, 8), textcoords="offset points", fontsize=8, ha="right")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(1, 200)
ax.set_xlabel("khoảng huấn luyện lại T (ngày, thang log)")
ax.set_ylabel("chi phí mỗi ngày (nghìn đô la, thang log)")
ax.set_title("Phương trình 8 với tham số bảng 9: đáy chữ U nằm gần một ngày")
plt.tight_layout()
plt.show()

Xấp xỉ của sách lệch dưới một phần trăm so với cực tiểu chính xác, và hai mức chi phí mỗi ngày gần
như bằng nhau: đáy chữ U rất phẳng. Hai bên đáy thì không phẳng. Bảng 8 của chương lấy hằng tuần làm
lịch minh hoạ cho phát hiện gian lận; với chính các tham số của bảng 9, lịch đó tốn gấp hơn ba lần
mức tối ưu mỗi ngày. Hai bảng không mâu thuẫn, vì sách gọi lịch của bảng 8 là điểm khởi đầu phải
hiệu chuẩn, nhưng phép tính cho thấy hiệu chuẩn đáng giá bao nhiêu.

Sách còn nói: nếu $T^*$ ngắn hơn thời gian triển khai thủ công của nhóm, hệ thống đang ở trạng thái
nợ kỹ thuật vĩnh viễn. Hình trên đọc được điều đó bằng tiền: một nhóm cần ba ngày để triển khai bằng
tay buộc phải đứng ở $T \ge 3$, bên phải đáy.

## 4. Bảng 10, và một đường cong duy nhất

Bảng 10: dưới xấp xỉ căn bậc hai, chi phí huấn luyện lại tăng 4 lần thì khoảng dài gấp đôi; lưu
lượng hay tốc độ suy giảm tăng 4 lần thì khoảng ngắn đi một nửa.

**Dự đoán:** cực tiểu chính xác có cho đúng gấp đôi và một nửa không?

In [ ]:
BON = sach(4, nguon="4 lần → 2 lần")
thay_doi = {"chi phí huấn luyện lại": dict(c=BON * C), "lưu lượng": dict(q=BON * Q),
            "tốc độ suy giảm": dict(gamma=BON * GAMMA)}
for ten, doi in thay_doi.items():
    ti_a = T_xap_xi(**doi) / T_a
    ti_e = T_chinh_xac(**doi) / T_e
    gia_tri = ti_a if ti_a > 1 else 1 / ti_a
    theo_sach(f"{ten} × 4, khoảng đổi (lần, xấp xỉ)", gia_tri, sach=2, nguon="4 lần → 2 lần")
    print(f"   cực tiểu chính xác: khoảng × {ti_e:.4f}")

Gần đúng, không đúng hẳn, và sai lệch có hướng. Lý do nằm ở chỗ chỉ một tổ hợp của các tham số quan
trọng. Đặt $x = \gamma T$ và $k = C\gamma / (Q V \text{Acc}_0)$. Chia phương trình 8 cho
$Q V \text{Acc}_0$ thì chi phí mỗi ngày thành

$$g(x) = 1 - \frac{1 - e^{-x}}{x} + \frac{k}{x}$$

nên $\gamma T^*$ chỉ phụ thuộc vào $k$, và xấp xỉ của sách là $\gamma T^* \approx \sqrt{2k}$. Mọi bộ
tham số nằm trên **một** đường cong. Với $k \ge 1$, $g$ giảm mãi khi $x$ tăng, nên không có cực tiểu
nào: mô hình nói không bao giờ đáng huấn luyện lại, vì một lần chạy đắt hơn toàn bộ giá trị mất trong
một thang thời gian $1/\gamma$.

In [ ]:
def x_chinh_xac(k):
    g = lambda u: 1 + np.expm1(-np.exp(u)) / np.exp(u) + k / np.exp(u)
    kq = optimize.minimize_scalar(g, bounds=(np.log(1e-6), np.log(1e3)), method="bounded", options={"xatol": 1e-12})
    return float(np.exp(kq.x))


k_bang9 = C * GAMMA / (Q * V * ACC0)
assert abs(x_chinh_xac(k_bang9) / GAMMA - T_e) < 1e-6   # cùng một cực tiểu, tính theo hai cách
cac_k = np.logspace(-5, np.log10(0.95), 60)
ti_le = np.array([x_chinh_xac(k) / np.sqrt(2 * k) for k in cac_k])

fig, ax = plt.subplots(figsize=(8, 3.9))
ax.semilogx(np.sqrt(2 * cac_k), ti_le, color=MAU[0], lw=2.2)
ax.axhline(1, color=XAM, lw=0.9, ls=":")
ax.plot([np.sqrt(2 * k_bang9)], [x_chinh_xac(k_bang9) / np.sqrt(2 * k_bang9)], "o", color=MAU[1], ms=8, zorder=5)
ax.annotate(f"bảng 9: γT* ≈ {np.sqrt(2 * k_bang9):.3f}", (np.sqrt(2 * k_bang9), x_chinh_xac(k_bang9) / np.sqrt(2 * k_bang9)),
            xytext=(4, 22), textcoords="offset points", fontsize=8,
            arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
for muc in (1.05, 1.25):
    x_muc = np.sqrt(2 * cac_k[np.argmax(ti_le >= muc)])
    ax.axvline(x_muc, color=XAM, lw=0.8, ls="--")
    ax.text(x_muc * 1.06, 2.2, f"lệch {muc - 1:.0%} từ γT* ≈ {x_muc:.2f}", fontsize=8, color="0.3", rotation=90, va="top")
ax.set_xlabel("γ · T* theo xấp xỉ căn bậc hai (thang log)")
ax.set_ylabel("T* chính xác / T* xấp xỉ")
ax.set_title("Mọi bộ tham số nằm trên một đường: xấp xỉ đúng khi γT* nhỏ")
ax.set_ylim(0.95, 2.4)
plt.tight_layout()
plt.show()

print("ví dụ, mô hình lớn hàng chục nghìn đô la mỗi lần chạy (C = 50,000, giả định):")
for gamma in (0.02, 0.1, 0.3):
    print(f"  γ = {gamma:<4}: xấp xỉ {T_xap_xi(c=50_000, gamma=gamma):5.2f} ngày,"
          f" chính xác {T_chinh_xac(c=50_000, gamma=gamma):5.2f} ngày")

Xấp xỉ luôn cho khoảng **ngắn hơn** cực tiểu chính xác, vì nó coi chi phí lỗi thời tăng theo $T^2$
mãi, trong khi thật ra độ chính xác không thể giảm dưới 0 nên phần mất mỗi ngày có trần. Sai lệch
dưới 5 phần trăm khi $\gamma T^*$ còn nhỏ cỡ 0.1, và lớn dần khi một lần chạy đắt hay thế giới đổi
nhanh. Đó đúng là điều kiện $\gamma T \ll 1$ mà sách đặt cho xấp xỉ.

## 5. Hình 8: theo lịch hay theo kích hoạt

Hình 8 của chương, đường cong tài sản mục rữa, đặt hai chính sách lên một đường suy giảm hàm mũ minh
hoạ. Theo lịch, độ chính xác được đặt lại mỗi 90 ngày, dù mô hình còn khoẻ hay đã rơi dưới ngưỡng.
Theo kích hoạt, mô hình được đặt lại khi chạm ngưỡng, nên giữ được một mức sàn cao hơn. Sách nói rõ
tốc độ suy giảm và các khoảng thời gian ở đây là minh hoạ.

Hình của sách giả định chính sách kích hoạt **thấy** ngay lúc độ chính xác chạm ngưỡng. Mô phỏng dưới
đây bỏ giả định đó, theo tinh thần phần C của phòng thí nghiệm đi kèm chương. Mọi tham số là giả định
của sổ tay: một năm 360 ngày với tốc độ suy giảm đổi theo mùa và một đợt dịch chuyển nhanh từ ngày
240 tới 270; ngưỡng 0.90; mỗi ngày 200 nhãn kết quả về, trễ 3 ngày; độ chính xác đo trên cửa sổ 7
ngày; một lần huấn luyện lại mất 2 ngày mới lên sản xuất.

**Dự đoán:** chính sách nào huấn luyện lại nhiều lần hơn? Chính sách kích hoạt có còn giữ được mức sàn
đúng bằng ngưỡng không?

In [ ]:
SO_NGAY = 360
toc_do = np.full(SO_NGAY, 0.0003)        # giả định: mùa rất yên
toc_do[120:240] = 0.001                  # mùa trôi vừa
toc_do[240:270] = 0.008                  # một đợt dịch chuyển nhanh
toc_do[270:] = 0.001
NGUONG = 0.90
CHU_KY = sach(90, nguon="90 ngày")


def mo_phong(chinh_sach, nhan_ngay=200, tre_nhan=3, cua_so=7, tre_hl=2, seed=SEED):
    """Một năm, từng ngày. Trả về độ chính xác thật, độ chính xác đo được, ngày lên sản xuất và lúc kích hoạt."""
    rng = np.random.default_rng(seed)
    log_acc, len_sx, cho = np.log(ACC0), 0, None
    acc, do_duoc, dung = np.empty(SO_NGAY), np.full(SO_NGAY, np.nan), np.zeros(SO_NGAY)
    lan, kich = [], []
    for t in range(SO_NGAY):
        if cho == t:                                     # mô hình mới lên sản xuất
            log_acc, len_sx, cho = np.log(ACC0), t, None
            lan.append(t)
        acc[t] = np.exp(log_acc)
        dung[t] = rng.binomial(nhan_ngay, acc[t])        # nhãn của hôm nay, sẽ về sau tre_nhan ngày
        den = t - tre_nhan                               # ngày cuối cùng đã có nhãn
        tu = max(len_sx, den - cua_so + 1)               # chỉ tính nhãn của mô hình đang chạy
        if den - tu + 1 >= cua_so:
            do_duoc[t] = dung[tu:den + 1].sum() / (nhan_ngay * cua_so)
        if cho is None:
            if chinh_sach == "lịch":
                bat = (t + tre_hl) % CHU_KY == 0 and t + tre_hl > 0
            elif chinh_sach == "lý tưởng":               # như hình 8: thấy trước độ chính xác thật của ngày mai
                bat = acc[t] * np.exp(-toc_do[t]) < NGUONG
            else:
                bat = not np.isnan(do_duoc[t]) and do_duoc[t] < NGUONG
            if bat:
                cho = t + tre_hl
                kich.append((t, acc[t]))
        log_acc -= toc_do[t]
    return acc, do_duoc, lan, kich


def tom_tat(acc, lan, kich):
    return {"lần huấn luyện lại": len(lan), "độ chính xác trung bình": round(float(acc.mean()), 4),
            "thấp nhất": round(float(acc.min()), 4), "số ngày dưới ngưỡng": int((acc < NGUONG).sum()),
            "kích hoạt khi còn khoẻ": int(sum(a >= NGUONG for _, a in kich))}


ket_qua = {}
for ten, kw in (("lịch", {}), ("lý tưởng", {"tre_hl": 1}), ("kích hoạt", {})):
    ket_qua[ten] = mo_phong(ten, **kw)
    r = tom_tat(ket_qua[ten][0], ket_qua[ten][2], ket_qua[ten][3])
    if ten == "lý tưởng":
        r.pop("kích hoạt khi còn khoẻ")   # nó cố ý kích hoạt ngay trước khi chạm ngưỡng
    print(f"{ten:<10}", r)

In [ ]:
ngay = np.arange(SO_NGAY)
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(10, 6.2), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for (ten, (acc, do_duoc, lan, kich)), mau, nhan in zip(ket_qua.items(), (MAU[1], XAM, MAU[0]),
                                                      ("theo lịch, mỗi 90 ngày", "kích hoạt lý tưởng, như hình 8",
                                                       "kích hoạt theo nhãn trễ 3 ngày")):
    ax.plot(ngay, acc, color=mau, lw=1.4 if ten == "lý tưởng" else 2.0, ls=":" if ten == "lý tưởng" else "-",
            label=nhan)
acc_k, do_k = ket_qua["kích hoạt"][0], ket_qua["kích hoạt"][1]
ax.plot(ngay, do_k, ".", color=MAU[0], ms=2.5, alpha=0.6, label="độ chính xác đo được, cửa sổ 7 ngày")
ax.axhline(NGUONG, color="0.2", lw=0.9, ls="--")
ax.text(3, NGUONG + 0.004, "ngưỡng 0.90 (giả định)", fontsize=8, color="0.2")
ax.axvspan(240, 270, color=MAU[3], alpha=0.15, lw=0)
ax.text(255, 0.715, "dịch chuyển\nnhanh", fontsize=8, color="0.3", ha="center", va="bottom")
ax.set_ylim(0.70, 0.96)
ax.set_ylabel("độ chính xác thật")
ax.set_title("Hình 8, mô phỏng lại: lịch cố định so với kích hoạt theo ngưỡng (mọi tham số là giả định)")
ax.legend(loc="lower left", fontsize=8, frameon=False, ncol=2)
ax2.step(ngay, toc_do * 1000, where="post", color="0.3", lw=1.4)
ax2.set_ylabel("γ (phần nghìn\nmỗi ngày)")
ax2.set_xlabel("ngày")
ax2.set_xlim(0, SO_NGAY)
plt.tight_layout()
plt.show()

Ba điều đọc được từ hình, và chúng giữ nguyên khi bạn đổi hạt giống:

* **Theo lịch vừa phí vừa muộn**, đúng như sách nói. Ở mùa yên, lần đặt lại ở ngày 90 đến khi mô
  hình còn trên ngưỡng. Ở đợt dịch chuyển, mô hình rơi sâu dưới ngưỡng và nằm đó tới lần chạy theo lịch.
* **Kích hoạt lý tưởng giữ mức sàn đúng bằng ngưỡng**, như hình 8, nhưng chỉ vì nó thấy trước độ chính
  xác thật và đưa mô hình mới lên ngay hôm sau.
* **Kích hoạt theo nhãn thật vẫn để lọt những vết lõm dưới ngưỡng**: nhãn về trễ 3 ngày, cửa sổ 7 ngày
  làm phép đo chậm thêm, và mô hình mới cần 2 ngày để lên sản xuất, nên khi độ chính xác thật rơi qua
  ngưỡng, nó thường nằm dưới đó vài ngày rồi mới được đặt lại. Đợt dịch chuyển càng nhanh thì vết lõm
  càng sâu.

Ô dưới quét hai tham số chính của chính sách kích hoạt: độ trễ của nhãn và số nhãn mỗi ngày. Mỗi
điểm là trung bình của 20 năm mô phỏng với 20 hạt giống khác nhau.

In [ ]:
HAT = range(20)


def trung_binh(**kw):
    kq = [tom_tat(*[mo_phong("kích hoạt", seed=s, **kw)[i] for i in (0, 2, 3)]) for s in HAT]
    return {k: float(np.mean([r[k] for r in kq])) for k in kq[0]}


cac_tre = [0, 1, 3, 7, 14]
cac_nhan = [20, 50, 200, 1000]
quet_tre = [trung_binh(tre_nhan=d) for d in cac_tre]
quet_nhan = [trung_binh(nhan_ngay=n) for n in cac_nhan]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(cac_tre, [r["số ngày dưới ngưỡng"] for r in quet_tre], "o-", color=MAU[0], lw=2)
ax1.set_xlabel("độ trễ của nhãn (ngày), 200 nhãn mỗi ngày")
ax1.set_ylabel("số ngày dưới ngưỡng mỗi năm")
ax1.set_title("Nhãn càng trễ, vết lõm càng sâu và dài")
ax1.set_ylim(0, None)
ax2.plot(cac_nhan, [r["kích hoạt khi còn khoẻ"] for r in quet_nhan], "o-", color=MAU[1], lw=2,
         label="kích hoạt khi còn khoẻ (báo nhầm)")
ax2.plot(cac_nhan, [r["lần huấn luyện lại"] for r in quet_nhan], "s--", color=MAU[0], lw=1.6,
         label="lần huấn luyện lại")
ax2.set_xscale("log")
ax2.set_xticks(cac_nhan, [str(n) for n in cac_nhan])
ax2.set_xlabel("số nhãn mỗi ngày, trễ 3 ngày")
ax2.set_ylabel("số lần mỗi năm")
ax2.set_title("Ít nhãn thì phép đo nhiễu: báo nhầm nhiều hơn")
ax2.set_ylim(0, None)
ax2.legend(loc="upper right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()
for d, r in zip(cac_tre, quet_tre):
    print(f"trễ {d:>2} ngày: {r['số ngày dưới ngưỡng']:5.1f} ngày dưới ngưỡng, sàn trung bình {r['thấp nhất']:.3f}")

Hai đòn bẩy kéo về hai phía. Nhãn về nhanh hơn rút ngắn thời gian mô hình nằm dưới ngưỡng. Nhiều nhãn
hơn làm phép đo ít nhiễu hơn, nên ít lần kích hoạt khi mô hình còn khoẻ, tức ít lần báo nhầm phải
điều tra. Đó là lý do sách nói huấn luyện lại theo kích hoạt cần số liệu theo dõi chắc chắn và hiệu
chuẩn cẩn thận để tránh báo nhầm hay bỏ sót. Sổ tay 03 tính xem cần bao nhiêu nhãn để thấy một mức
giảm nhất định.

## 6. Huấn luyện lại tự động không người duyệt

Một ngộ nhận cuối chương cho rằng huấn luyện lại tự động bảo đảm hiệu năng tối ưu mà không cần con
người. Ví dụ của sách: một hệ gợi ý tin tức huấn luyện lại trên dữ liệu cuối tuần có thể kém đi vào
ngày thường, vì hành vi người dùng hai bối cảnh khác hẳn nhau. Sách cũng cảnh báo về những mô hình
qua được kiểm định tổng hợp mà vẫn suy giảm ở các trường hợp biên.

Thí nghiệm dưới đây dựng lại tình huống đó trên bộ chữ số `load_digits` (1,797 ảnh 8 × 8). Mô hình
đang chạy học trên 900 ảnh với đủ mười chữ số. "Cuối tuần" là một cửa sổ dữ liệu mới gần như chỉ có
chữ số 0 tới 4, mỗi chữ số 5 tới 9 chỉ còn 3 ảnh. Pipeline tự động huấn luyện lại trên cửa sổ mới
nhất, kiểm định trên một phần giữ riêng **của chính cửa sổ đó**, và nâng cấp nếu độ chính xác kiểm
định từ 0.90 trở lên. Phân phối cuối tuần và ngưỡng 0.90 là giả định của sổ tay.

**Dự đoán:** mô hình mới qua cổng kiểm định không? Vào thứ Hai, khi lưu lượng trở về đủ mười chữ số,
nó đúng bao nhiêu phần trăm?

In [ ]:
X, y = load_digits(return_X_y=True)
X = X / 16.0
rng = np.random.default_rng(SEED)
thu_tu = rng.permutation(len(X))
tap_cu, con_lai = thu_tu[:900], thu_tu[900:]
ngay_thuong, kho = con_lai[:450], con_lai[450:]           # lưu lượng ngày thường: đủ mười chữ số
it = np.isin(y[kho], [5, 6, 7, 8, 9])
chon_it = np.concatenate([kho[it][y[kho[it]] == k][:3] for k in range(5, 10)])
cuoi_tuan = rng.permutation(np.concatenate([kho[~it], chon_it]))
cat = int(0.7 * len(cuoi_tuan))
ct_hl, ct_kd = cuoi_tuan[:cat], cuoi_tuan[cat:]
print("số ảnh mỗi chữ số trong cửa sổ cuối tuần:", np.bincount(y[cuoi_tuan], minlength=10).tolist())


def huan_luyen(chi_so):
    return LogisticRegression(max_iter=2000).fit(X[chi_so], y[chi_so])


def dung(mo_hinh, chi_so):
    return float((mo_hinh.predict(X[chi_so]) == y[chi_so]).mean())


cu = huan_luyen(tap_cu)
moi = huan_luyen(ct_hl)
NGUONG_KD = 0.90   # giả định của sổ tay
qua_cong = dung(moi, ct_kd) >= NGUONG_KD
do_tren_may("mô hình mới, kiểm định trên cửa sổ cuối tuần", 100 * dung(moi, ct_kd), "%")
print(f"cổng tự động (từ {NGUONG_KD:.2f} trở lên trên cửa sổ mới nhất): {'cho qua' if qua_cong else 'chặn'}")

nt_it = ngay_thuong[np.isin(y[ngay_thuong], [5, 6, 7, 8, 9])]
for ten, m in (("mô hình đang chạy", cu), ("mô hình mới", moi)):
    do_tren_may(f"{ten}, thứ Hai, đủ mười chữ số", 100 * dung(m, ngay_thuong), "%")
    do_tren_may(f"{ten}, thứ Hai, riêng chữ số 5 tới 9", 100 * dung(m, nt_it), "%")

Cổng tự động nhìn đúng dữ liệu mà mô hình mới đã quen, nên nó không có cách nào thấy chữ số 5 tới 9
đang bị bỏ quên. Hai cách sửa, cả hai đều là những gì sách đòi hỏi:

* **Một tập tham chiếu cố định** có phân phối của lưu lượng thật, cùng một mức sàn **theo từng lớp**
  (phân tích theo lát cắt, chứ không chỉ một con số tổng hợp), và so với mô hình đang chạy trên cùng
  tập đó. Sách gọi việc đánh giá ứng viên trong điều kiện y hệt nhau là tiền đề của quyết định nâng
  cấp đúng.
* **Không huấn luyện lại chỉ trên cửa sổ mới nhất** khi cửa sổ ấy lệch: gộp nó với dữ liệu cũ.

Ô dưới thử cổng thứ nhất trên mô hình mới, và thử cách thứ hai. Mức sàn 0.85 cho mỗi lớp là giả định.

In [ ]:
SAN_LOP = 0.85   # giả định của sổ tay


def cong_tham_chieu(mo_hinh, dang_chay, tham_chieu=ngay_thuong):
    tung_lop = [dung(mo_hinh, tham_chieu[y[tham_chieu] == k]) for k in range(10)]
    khong_kem = dung(mo_hinh, tham_chieu) >= dung(dang_chay, tham_chieu) - 0.01
    return khong_kem and min(tung_lop) >= SAN_LOP, min(tung_lop)


ok_moi, san_moi = cong_tham_chieu(moi, cu)
gop = huan_luyen(np.concatenate([tap_cu, ct_hl]))
ok_gop, san_gop = cong_tham_chieu(gop, cu)
print(f"cổng tham chiếu, mô hình mới: {'cho qua' if ok_moi else 'chặn'} (lớp kém nhất đúng {san_moi:.0%})")
print(f"cổng tham chiếu, mô hình học trên dữ liệu cũ gộp cửa sổ mới: {'cho qua' if ok_gop else 'chặn'}"
      f" (lớp kém nhất đúng {san_gop:.0%})")
do_tren_may("mô hình gộp, kiểm định trên cửa sổ cuối tuần", 100 * dung(gop, ct_kd), "%")
do_tren_may("mô hình gộp, thứ Hai, đủ mười chữ số", 100 * dung(gop, ngay_thuong), "%")

Đây là lý do sách đòi quy trình leo thang cho kết quả kiểm định bất thường, người duyệt cho những mẫu
số liệu lạ, và quyền ghi đè khi tự động hoá cho ra kết quả đáng ngờ. Một pipeline tự động chỉ an toàn
bằng cổng kiểm định của nó, và một cổng chỉ nhìn dữ liệu mới nhất thì đang kiểm định đúng cái giả định
cần phải nghi ngờ.

## Thử thêm

1. Ở phần 3, đặt `C = 50_000`, mức "hàng chục nghìn đô la mỗi lần chạy" mà sách nói cho mô hình lớn.
   $T^*$ dài ra bao nhiêu lần, và đúng như bảng 10 dự đoán không?
2. Ở phần 5, đổi `toc_do[240:270]` thành 0.02. Chính sách theo lịch rơi xuống bao nhiêu trước lần đặt
   lại kế tiếp? Rồi đổi `NGUONG` thành 0.92 cho chính sách kích hoạt: số lần huấn luyện lại và số lần
   báo nhầm đổi thế nào?
3. Ở phần 6, cho mỗi chữ số 5 tới 9 trong cửa sổ cuối tuần 20 ảnh thay vì 3. Cổng tự động và cổng tham
   chiếu còn bất đồng không?

## Tóm lại

* Ở 1 triệu giao dịch mỗi ngày và 0.005 đô la mỗi điểm phần trăm mỗi truy vấn, một điểm đáng 5,000 đô
  la mỗi ngày, nên phương trình 3 mở rất nhanh; nó vẫn cần bằng chứng về kết quả, không chỉ tín hiệu trôi.
* Chi phí mỗi ngày là một đường chữ U; với bảng 9, đáy nằm gần 1 ngày, xấp xỉ căn bậc hai lệch dưới một
  phần trăm, còn lịch hằng tuần tốn gấp hơn ba lần mỗi ngày.
* Mọi bộ tham số nằm trên một đường cong của $k = C\gamma/(QV\text{Acc}_0)$; xấp xỉ đúng khi
  $\gamma T^*$ nhỏ, luôn cho khoảng ngắn hơn cực tiểu thật, và khi $k \ge 1$ thì không có cực tiểu nào.
* Theo lịch vừa phí vừa muộn; theo kích hoạt giữ sàn cao hơn, nhưng sàn ấy thấp dần theo độ trễ của
  nhãn, và số nhãn ít làm nó báo nhầm.
* Một cổng kiểm định chỉ nhìn cửa sổ dữ liệu mới nhất có thể cho qua một mô hình kém hẳn trên lưu lượng
  thật; cần tập tham chiếu cố định và mức sàn theo từng lát cắt.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Vận hành học máy — khi hệ thống sẵn sàng mà vẫn sai](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch14-ml-ops/), dựng từ chương
[*ML Operations*](https://mlsysbook.ai/vol1/ml_ops/ml_ops.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_14_ml_ops.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.